# 5.2 SGLang: RadixAttention & Structured Generation Lab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/06_engines/05.2_sglang/lab.ipynb)
[![Open In Molab](https://raw.githubusercontent.com/marimo-team/marimo/main/docs/_static/molab-badge.svg)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/content/06_engines/05.2_sglang/lab.ipynb)

This lab demonstrates SGLang's two key innovations:
1. RadixAttention prefix cache hit rate advantage over hash-based caching
2. Jump-forward optimization for structured (JSON/regex) generation

**Prerequisites**: SGLang server running on port 30000:
```bash
python -m sglang.launch_server --model-path meta-llama/Meta-Llama-3-8B-Instruct --port 30000
```

In [ ]:
# ============================================================
# CELL: Install dependencies via subprocess
# Uses subprocess to ensure compatibility with Colab and Molab
# ============================================================
import subprocess  # For running pip install as a subprocess
import sys  # For getting the current Python executable path

# Install openai client (SGLang exposes OpenAI-compatible API)
# Install numpy for statistical calculations on latency data
# Install matplotlib for generating comparison charts
subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '-q', 'openai', 'numpy', 'matplotlib']
)

In [ ]:
# ============================================================
# CELL: Imports and server connection
# All imports for the entire notebook are here
# ============================================================
import time  # For high-precision latency measurements
import json  # For parsing JSON structured outputs
import numpy as np  # For computing mean, median, percentiles
import matplotlib.pyplot as plt  # For plotting comparison charts
import openai  # OpenAI-compatible client works with SGLang

# SGLang server URL (must be running separately)
SGLANG_URL = "http://localhost:30000/v1"

# Create client pointing at the local SGLang server
# api_key="none" because SGLang doesn't require auth locally
client = openai.OpenAI(base_url=SGLANG_URL, api_key="none")

## Experiment 1: RadixAttention Prefix Reuse

We measure latency across repeated requests sharing the same system prompt.
RadixAttention caches the prefix KV tensors in a radix tree, so subsequent
requests skip prefill for shared tokens entirely.

In [ ]:
# ============================================================
# CELL: Configure experiment parameters
# Change these values and re-run to see different behaviors
# ============================================================

# Number of requests to warm the radix tree cache
N_WARMUP = 2
# Number of requests to measure after cache is populated
N_MEASURE = 8
# Max tokens to generate per response (controls decode time)
MAX_TOKENS = 100

# Shared system prompt: cached by RadixAttention after first use
# All subsequent requests reuse these KV tensors from the tree
SYSTEM_PROMPT = (
    "You are an expert ML engineer specializing in LLM inference optimization. "
    "Provide concise, technically accurate answers about GPU memory management, "
    "KV cache strategies, batching algorithms, and serving architectures. "
    "Always include specific numbers and formulas where applicable."
)

# Diverse user queries that all share the system prompt prefix
# RadixAttention reuses the system prompt KV cache for each one
QUERIES = [
    "How much KV cache memory does Llama-3 70B need per token?",
    "What is the roofline model for transformer inference?",
    "Explain continuous batching vs static batching.",
    "How does GQA reduce memory compared to MHA?",
    "What batch size saturates an A100 for 7B decode?",
    "Compare PagedAttention vs RadixAttention.",
    "What is the arithmetic intensity of prefill vs decode?",
    "How does speculative decoding improve latency?",
    "What is the memory bandwidth utilization during decode?",
    "Explain chunked prefill and its benefits.",
]

In [ ]:
# ============================================================
# CELL: Run cold vs warm latency comparison
# Cold = prefix not in cache; Warm = prefix cached in radix tree
# ============================================================

def measure_latency(query: str) -> float:
    """Send one request and return end-to-end latency in ms."""
    # Record precise start time
    t0 = time.perf_counter()
    # Send request with shared system prompt (prefix) + unique query
    client.chat.completions.create(
        model="default",
        messages=[
            # System prompt becomes the shared prefix in the radix tree
            {"role": "system", "content": SYSTEM_PROMPT},
            # User query is the unique suffix per request
            {"role": "user", "content": query},
        ],
        max_tokens=MAX_TOKENS,
        # temperature=0 for deterministic output (fair comparison)
        temperature=0,
    )
    # Return elapsed time in milliseconds
    return (time.perf_counter() - t0) * 1000


# --- COLD PASS: prefix not yet in the radix tree ---
# First time seeing this system prompt, must compute full prefill
cold_latencies = []
for q in QUERIES[:N_WARMUP + N_MEASURE]:
    # Each request triggers full prefill of system prompt tokens
    cold_latencies.append(measure_latency(q))

# --- WARM PASS: prefix now cached in the radix tree ---
# RadixAttention finds the longest matching prefix and reuses KV
warm_latencies = []
for q in QUERIES[:N_WARMUP + N_MEASURE]:
    # Only the unique query suffix needs prefill; system prompt reused
    warm_latencies.append(measure_latency(q))

# Separate warmup from measurement for accurate statistics
cold_measured = cold_latencies[N_WARMUP:]  # Discard warmup samples
warm_measured = warm_latencies[N_WARMUP:]  # Discard warmup samples

# Display summary statistics
print("RadixAttention Prefix Reuse Results")
print("=" * 50)
print(f"{'Pass':<8} {'Mean ms':<10} {'P50 ms':<10} {'P95 ms':<10}")
for name, lats in [("Cold", cold_measured), ("Warm", warm_measured)]:
    # Convert to numpy array for percentile calculations
    arr = np.array(lats)
    print(f"{name:<8} {arr.mean():<10.1f} {np.median(arr):<10.1f} {np.percentile(arr, 95):<10.1f}")

# Calculate and report the speedup from prefix caching
speedup = np.mean(cold_measured) / np.mean(warm_measured)
print(f"\nPrefix cache speedup: {speedup:.2f}x")

In [ ]:
# ============================================================
# CELL: Visualize cold vs warm latency as bar chart
# Shows the per-request improvement from RadixAttention caching
# ============================================================
fig, ax = plt.subplots(1, 1, figsize=(8, 4))

# X-axis positions for side-by-side bars
x = np.arange(N_MEASURE)
# Bar width for side-by-side comparison
width = 0.35

# Cold latencies in rose color (slower, no cache)
ax.bar(x - width/2, cold_measured, width,
       label='Cold (no cache)', color='#ffe4e6', edgecolor='#000')
# Warm latencies in green color (faster, prefix cached)
ax.bar(x + width/2, warm_measured, width,
       label='Warm (RadixAttention)', color='#dcfce7', edgecolor='#000')

# Label axes clearly
ax.set_xlabel('Request Index')
ax.set_ylabel('Latency (ms)')
ax.set_title('RadixAttention: Cold vs Warm Prefix Cache')
ax.set_xticks(x)
ax.legend()
# Light grid for readability
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
# Save chart to disk for inclusion in reports
plt.savefig('radix_attention_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: radix_attention_comparison.png")

## Experiment 2: Structured Generation with Jump-Forward

SGLang's jump-forward optimization skips tokens that are deterministic given
the grammar (JSON delimiters, field names, colons, quotes). We compare
unconstrained vs JSON-schema-constrained generation latency.

In [ ]:
# ============================================================
# CELL: Configure structured generation benchmark
# Defines the JSON schema and test input
# ============================================================

# Number of trials per condition (unconstrained vs constrained)
N_TRIALS = 5

# JSON schema requiring 4 specific fields
# Many tokens in this output are deterministic (braces, quotes, colons)
# SGLang's jump-forward skips all deterministic tokens in one step
ENTITY_SCHEMA = {
    "type": "object",
    "properties": {
        # String field for person's name
        "name": {"type": "string"},
        # String field for job title
        "role": {"type": "string"},
        # String field for organization
        "company": {"type": "string"},
        # Numeric confidence score 0-1
        "confidence": {"type": "number"},
    },
    # All fields are mandatory in the output
    "required": ["name", "role", "company", "confidence"],
}

# Test text for entity extraction task
TEST_TEXT = "Sam Altman, CEO of OpenAI, announced new partnerships with enterprise clients."
# Prompt combining instruction + input text
PROMPT = f"Extract the person entity from this text as JSON: {TEST_TEXT}"

In [ ]:
# ============================================================
# CELL: Benchmark unconstrained vs constrained generation
# Demonstrates jump-forward making constrained FASTER
# ============================================================

def bench_unconstrained() -> float:
    """Measure latency for free-form generation (no schema)."""
    # Start timing before the API call
    t0 = time.perf_counter()
    # Generate without constraints; model produces all tokens sequentially
    client.chat.completions.create(
        model="default",
        messages=[{"role": "user", "content": PROMPT + " Respond in JSON."}],
        max_tokens=150,
        temperature=0,  # Deterministic for fair measurement
    )
    # Return elapsed time in milliseconds
    return (time.perf_counter() - t0) * 1000


def bench_constrained() -> float:
    """Measure latency for JSON-schema generation (jump-forward active)."""
    # Start timing before the API call
    t0 = time.perf_counter()
    # Generate with JSON schema; SGLang compiles schema to FSA
    # and uses jump-forward to skip deterministic tokens
    resp = client.chat.completions.create(
        model="default",
        messages=[{"role": "user", "content": PROMPT}],
        # Specify JSON schema constraint for output format
        response_format={
            "type": "json_schema",
            "json_schema": {"name": "entity", "schema": ENTITY_SCHEMA},
        },
        max_tokens=150,
        temperature=0,  # Deterministic for fair measurement
    )
    # Compute elapsed time
    elapsed = (time.perf_counter() - t0) * 1000
    # Verify output is valid JSON matching our schema
    parsed = json.loads(resp.choices[0].message.content)
    # Assert required field exists to confirm schema enforcement worked
    assert "name" in parsed, "Schema enforcement failed: missing 'name' field"
    return elapsed


# Run N_TRIALS of each condition
# Unconstrained: model generates every token sequentially
unconstrained_lats = [bench_unconstrained() for _ in range(N_TRIALS)]
# Constrained: jump-forward skips deterministic JSON tokens
constrained_lats = [bench_constrained() for _ in range(N_TRIALS)]

# Report comparison results
print("Structured Generation Benchmark")
print("=" * 45)
print(f"{'Mode':<15} {'Mean ms':<10} {'P50 ms':<10}")
for name, lats in [("Unconstrained", unconstrained_lats), ("JSON Schema", constrained_lats)]:
    # Convert to numpy array for statistical functions
    arr = np.array(lats)
    print(f"{name:<15} {arr.mean():<10.1f} {np.median(arr):<10.1f}")

# Compute ratio: values > 1.0 mean constrained is FASTER
# This happens because jump-forward skips ~60% of decode steps
ratio = np.mean(unconstrained_lats) / np.mean(constrained_lats)
print(f"\nConstrained/Unconstrained ratio: {ratio:.2f}x")
print("(Values > 1.0 mean jump-forward makes constrained FASTER)")

In [ ]:
# ============================================================
# CELL: Plot structured generation comparison
# Bar chart showing unconstrained vs constrained latency
# ============================================================
fig_7, ax_7 = plt.subplots(1, 1, figsize=(6, 4))

# Category labels for x-axis
categories = ['Unconstrained', 'JSON Schema\n(jump-forward)']
# Compute mean latency for each condition
means = [np.mean(unconstrained_lats), np.mean(constrained_lats)]
# Compute standard deviation for error bars
stds = [np.std(unconstrained_lats), np.std(constrained_lats)]
# Rose color for slow (unconstrained), green for fast (constrained)
colors = ['#ffe4e6', '#dcfce7']

# Draw bars with error bars showing 1 standard deviation
bars = ax_7.bar(categories, means, yerr=stds, capsize=5,
              color=colors, edgecolor='#000')

# Annotate each bar with its exact mean value
for bar, mean in zip(bars, means):
    ax_7.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'{mean:.0f}ms', ha='center', fontsize=11)

# Axis labels and title
ax_7.set_ylabel('Latency (ms)')
ax_7.set_title('SGLang Jump-Forward: Constrained vs Unconstrained')
# Light horizontal grid for readability
ax_7.grid(axis='y', alpha=0.3)

plt.tight_layout()
# Save chart for reports and presentations
plt.savefig('jump_forward_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: jump_forward_comparison.png")

## Experiment 3: Prefix Sharing Factor vs Memory

We simulate how KV cache memory scales with concurrent users when a common
system prompt is shared. RadixAttention stores the shared prefix exactly once
regardless of how many users reference it.

In [ ]:
# ============================================================
# CELL: Simulated prefix sharing model
# Compares memory with vs without RadixAttention tree sharing
# Uses Mistral-7B GQA parameters for realistic numbers
# ============================================================

# Typical system prompt length in tokens
SYSTEM_PROMPT_TOKENS = 512
# KV cache bytes per token for Mistral-7B:
# 8 KV heads * 128 dim * 32 layers * 2 (K+V) * 2 bytes = 131,072
KV_BYTES_PER_TOKEN = 131_072
# Average unique tokens per user (query + response)
UNIQUE_TOKENS_PER_USER = 200

# Simulate different concurrency levels
user_counts = [1, 10, 50, 100, 500, 1000]

# Lists to store memory usage for each approach
mem_no_sharing = []  # Naive: each user stores full prefix independently
mem_with_radix = []  # RadixAttention: shared prefix stored once

for n_users in user_counts:
    # WITHOUT sharing: every user duplicates the system prompt KV cache
    # Total = n_users * (system_prompt + unique_tokens) * bytes_per_token
    total_no_share = n_users * (SYSTEM_PROMPT_TOKENS + UNIQUE_TOKENS_PER_USER)
    # Convert bytes to gigabytes
    mem_no_sharing.append(total_no_share * KV_BYTES_PER_TOKEN / 1e9)

    # WITH RadixAttention: system prompt stored ONCE in the radix tree
    # Only unique per-user tokens are stored separately
    total_radix = SYSTEM_PROMPT_TOKENS + n_users * UNIQUE_TOKENS_PER_USER
    # Convert bytes to gigabytes
    mem_with_radix.append(total_radix * KV_BYTES_PER_TOKEN / 1e9)

# Calculate percentage savings at each concurrency level
savings = [(1 - r/n) * 100 for n, r in zip(mem_no_sharing, mem_with_radix)]

# Print formatted results table
print("KV Cache Memory: No Sharing vs RadixAttention")
print("=" * 55)
print(f"{'Users':<8} {'No Share (GB)':<15} {'Radix (GB)':<13} {'Savings':<10}")
for i, n in enumerate(user_counts):
    # Show memory for each concurrency level
    print(f"{n:<8} {mem_no_sharing[i]:<15.2f} {mem_with_radix[i]:<13.2f} {savings[i]:<10.1f}%")

In [ ]:
# ============================================================
# CELL: Plot memory scaling comparison (2 subplots)
# Left: absolute memory; Right: savings percentage
# ============================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# --- Left plot: absolute memory usage vs user count ---
# Red line for naive approach (linear scaling)
ax1.plot(user_counts, mem_no_sharing, 'o-',
         color='#991b1b', label='No Sharing', linewidth=2)
# Green line for RadixAttention (sub-linear scaling)
ax1.plot(user_counts, mem_with_radix, 's-',
         color='#166534', label='RadixAttention', linewidth=2)
# Shade the area between lines to highlight savings
ax1.fill_between(user_counts, mem_with_radix, mem_no_sharing,
                 alpha=0.1, color='#dcfce7')
# Label axes
ax1.set_xlabel('Concurrent Users')
ax1.set_ylabel('KV Cache Memory (GB)')
ax1.set_title('Memory Scaling: Shared System Prompt')
ax1.legend()
# Light grid for readability
ax1.grid(alpha=0.3)
# Reference line showing A100 80GB memory limit
ax1.axhline(y=80, color='gray', linestyle='--', alpha=0.5)
# Annotate the reference line
ax1.text(50, 82, 'A100 80GB limit', fontsize=9, color='gray')

# --- Right plot: memory savings percentage ---
# Bar chart showing savings grows with more users
ax2.bar(range(len(user_counts)), savings,
        color='#dbeafe', edgecolor='#000')
# Set x-tick labels to user counts
ax2.set_xticks(range(len(user_counts)))
ax2.set_xticklabels(user_counts)
# Label axes
ax2.set_xlabel('Concurrent Users')
ax2.set_ylabel('Memory Saved (%)')
ax2.set_title('RadixAttention Memory Savings')
# Light grid for readability
ax2.grid(axis='y', alpha=0.3)
# Annotate each bar with its percentage value
for i, s in enumerate(savings):
    ax2.text(i, s + 1, f'{s:.0f}%', ha='center', fontsize=10)

plt.tight_layout()
# Save combined figure for reports
plt.savefig('radix_memory_scaling.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: radix_memory_scaling.png")
# Print key takeaway for the reader
print(f"\nKey insight: At 1000 users, RadixAttention saves {savings[-1]:.0f}% of KV cache memory")
print(f"because the {SYSTEM_PROMPT_TOKENS}-token system prompt is stored exactly once.")